# Exercise 2.2.4 — Transforming Data & Creating New Features

This exercise reads the cleaned file produced by **Exercise 2.2.3** (`data/10_cleaned/datania_households_clean.csv`) and turns it into an **analysis-ready** dataset by creating derived columns.

You will practice:
- **Recoding**: per-capita values, age bands with `pd.cut()`, code->label maps with `map()`
- **Conditional assignment** with `np.where()` and `.loc[]`
- Clean, chainable column creation with `assign()` (including dependent columns via lambdas)
- Multi-way categories with `np.select()`
- Custom logic with `apply()` — named functions, lambdas, and row-wise `apply(axis=1)`
- Saving the feature table to `20_processed/`

> **Pipeline:** run Exercise 2.2.3 first so the cleaned file exists. This notebook writes to `20_processed/`.

### Path Setup (run first)

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_CLEANED_DIR = '../../data/10_cleaned'
FILE_NAME = 'datania_households_clean.csv'
clean_path = os.path.join(DATA_CLEANED_DIR, FILE_NAME)

df = pd.read_csv(clean_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

---

## Task 1 — Recode: per-capita, bands, and label maps

Recoding turns raw inputs into variables that are easier to analyse and report.

In [ ]:
# Income per household member (vectorised: column / column)
df['income_per_capita'] = # your code here

df[['hh_id', 'income_dkw', 'hh_size', 'income_per_capita']].head()

In [ ]:
# Age bands with pd.cut: Child (0-18), Adult (18-65), Elderly (65-120)
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 65, 120],
    labels= # your code here
)
df['age_group'].value_counts(dropna=False)

In [ ]:
# Map education codes to labels
education_map = {1: 'Primary', 2: 'Secondary', 3: 'Tertiary', 4: 'Higher levels'}
df['education_label'] = df['education_code']. # your code here

df['education_label'].value_counts(dropna=False)

**Questions:**

- `map()` returns `NaN` for any code not in the dictionary. Which households end up with a missing `education_label`, and why (think back to 2.2.3)?
- An income of `NaN` divided by `hh_size` gives what? Check `income_per_capita` for households whose income was missing.

---

## Task 2 — Conditional assignment with `np.where()` and `.loc[]`

`np.where()` is a vectorised IF: *if condition, value A, else value B*. `.loc[]` updates **existing** values that match a condition.

In [ ]:
# Classify by population density (a different signal than the reported urban_rural)
df['area_type'] = np.where(df['pop_density'] > 500, # your code here — 'Urban', 'Rural' )

df[['hh_id', 'pop_density', 'urban_rural', 'area_type']].head(8)

In [ ]:
# Bucket household size by updating an existing column with .loc[]
df['hh_category'] = 'Standard'
df.loc[df['hh_size'] >= 7, 'hh_category'] = # your code here — 'Large'
df.loc[df['hh_size'] <= 2, # your code here ] = 'Small'

df[['hh_id', 'hh_size', 'hh_category']].head(10)

**Questions:**

- Where the reported `urban_rural` and the density-based `area_type` disagree, which would you trust, and how would you investigate?
- `np.where()` treats a `NaN` population density as "not > 500" and labels it `Rural`. Is that the behaviour you want? How could you make the unknown explicit?

---

## Task 3 — Clean column creation with `assign()`

`assign()` returns a **new** DataFrame, which keeps your code in a readable chain. With lambdas it can even build a column that depends on one created earlier in the same call.

In [ ]:
# Create two independent columns at once
df = df.assign(
    income_thousands = df['income_dkw'] / 1000,
    pop_density_log = # your code here — logaritmic value
)
df[['hh_id', 'income_dkw', 'income_thousands', 'pop_density', 'pop_density_log']].head()

In [ ]:
# Dependent columns: high_income is derived from income_per_capita created in the same call
df = df.assign(
    income_per_capita = lambda x: x['income_dkw'] / x['hh_size'],
    high_income = lambda x: np.where( # your code here — x['income_per_capita'] > 25000, 'Yes', 'No' )
)
df[['hh_id', 'income_per_capita', 'high_income']].head()

**Question:** Why must `high_income` reference `lambda x: x[...]` instead of `df[...]`? What is `x` at that point in the chain?

---

## Task 4 — Multi-way categories with `np.select()`

For more than two outcomes, `np.select()` is cleaner than nested `np.where()`. It evaluates conditions **in order** and takes the first match; `default` handles everything else (including `NaN`).

In [ ]:
conditions = [
    df['income_dkw'] < 40000,
    df['income_dkw'] < 70000,
    df['income_dkw'] >= 70000,
]
choices = ['Low', 'Medium', 'High']

df['income_band'] = np.select(conditions, choices, default= # your code here — 'Unknown' )
df['income_band'].value_counts()

**Questions:**

- How many households fall into `Unknown`? What do they have in common?
- Why does the **order** of the conditions matter? What would happen if `>= 70000` came first?

---

## Task 5 — Custom logic with `apply()`

When built-in vectorised operations are not enough, `apply()` runs your own function on every value (or every row). Prefer vectorised code when you can — `apply()` is slower — but it is invaluable for complex logic.

In [ ]:
# A named function for multi-step logic
def classify_size(size):
    if size <= 2:
        return 'Small'
    elif size <= 5:
        return 'Medium'
    else:
        return 'Large'

df['hh_size_class'] = df['hh_size']. # your code here — apply(classify_size)
df['hh_size_class'].value_counts()

In [ ]:
# A lambda for simple one-line logic
df['high_income_flag'] = df['income_dkw'].apply(lambda x: # your code here — 'Yes' if x > 50000 else 'No' )
df[['hh_id', 'income_dkw', 'high_income_flag']].head()

In [ ]:
# A named function is clearer than a lambda once the logic has guards.
# apply(axis=1) passes one ROW at a time, so the function can read several columns.
def compute_income_per_capita(row):
    """Income per household member, or NaN if inputs are invalid."""
    income, hh_size = row['income_dkw'], row['hh_size']
    if pd.isna(income) or pd.isna(hh_size) or hh_size <= 0:
        return np.nan
    return round(income / hh_size, 2)

df['per_capita_safe'] = df.apply( # your code here — compute_income_per_capita, axis=1 )
df[['hh_id', 'income_dkw', 'hh_size', 'per_capita_safe']].head()

**Reusable cleaning functions + `apply`**

When a *new* messy extract arrives, you can wrap the cleaning rules from 2.2 in a reusable function and `apply()` it — the natural home for `clean_income` and `standardise_date`. The pipeline data is already clean here, so we demonstrate on sample values.

In [ ]:
# Task 5 — build the whole function yourself.
# clean_income: clean ONE raw income value and return a float (or NaN) —
# handle NaN, strip " "/"Ar"/",", map text codes ("unknown", "NA", ...) to NaN, then convert to a number.
def clean_income(value):
    # your code here — build the full function body
    return  # your cleaned value, or np.nan

sample_income = pd.Series(["Ar 32,000", "45 000", "unknown", "1,200,000", np.nan])
sample_income.apply(clean_income)

In [ ]:
# Build the whole function yourself.
# standardise_date: turn ONE survey-date string into a datetime (or NaT) —
# handle NaN / "not recorded"; accept MM/DD/YYYY and YYYY/MM/DD (use .split());
# fix an inverted month (the middle part > 12); then parse with pd.to_datetime.
def standardise_date(value):
    # your code here — build the full function body
    return  # a datetime, or pd.NaT

sample_dates = pd.Series(["03/15/2025", "2025/01/18", "2025-13-01", "not recorded", "2025-01-10"])
sample_dates.apply(standardise_date)

**Questions:**

- When is a **named function** better than a **lambda**? When is the lambda fine?
- The lambda flag treats `NaN > 50000` as `False` -> `'No'`. Is silently labelling unknown income as "not high" safe? How would you guard against it?

---

## Task 6 — Save the feature table to `20_processed/`

Cleaned data lives in `10_cleaned/`; derived/analysis tables go in `20_processed/`. Save the enriched DataFrame there for the merging step in 2.2.5.

In [ ]:
DATA_PROC_DIR = '../../data/20_processed'
os.makedirs(DATA_PROC_DIR, exist_ok=True)
out_path = os.path.join(DATA_PROC_DIR, 'datania_households_features.csv')

df.to_csv( # your code here — index=False )
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded:', check.shape)
check.head()

**Questions:**

- How many columns did you add compared with the cleaned input?
- Which of your new columns are **vectorised** (fast) and which used `apply()` (slower)? On a million-row file, which would you rewrite first?